# Lab 4 — Analyse the Uploader Experiment Correctly
# المختبر 4 — تحليل تجربة رافع المستندات بشكل صحيح

**Module 4 · الوحدة 4 — Analysis Pitfalls: Peeking and Multiple Testing / مزالق التحليل: المراقبة المتكررة والاختبارات المتعددة**
SDA-DSC-213 · Experimentation, A/B Testing and Causal Inference · SDAIA Academy

---

### The experiment is over. Now do not ruin it.

Lab 2 designed it. Lab 3 powered it. The data is in: 80,000 assigned users, a clean SRM,
a frozen analysis plan. Everything from here is a chance to manufacture a wrong answer
that looks right.

This lab walks you into **two traps on purpose** before it tells you they are traps.
You will get a number, believe it, and then find out it was diluted. You will get a
confidence interval, believe it, and then find out it was half as wide as it should be.
That sequence is the lesson — a correct analysis is not a formula you apply, it is a set
of questions you ask before you apply one.

### What you will build

| # | Step | Minutes |
|---|---|---|
| 1 | The SRM gate, then the **all-assigned-users** estimate (**trap 1**) | 8 |
| 2 | The **triggered-users** estimate; two-proportion vs OLS + HC3 | 8 |
| 3 | **CUPED / covariate adjustment** — the same estimate, a tighter interval | 8 |
| 4 | The **guardrail family** under Bonferroni | 8 |
| 5 | The **segment sweep** under Benjamini–Hochberg | 8 |
| 6 | **Trap 2**: session-level analysis without clustering | 8 |
| 7 | The **peeking simulation**: 5% → 25% | 10 |
| 8 | The decision paragraph: statistical vs practical significance | 8 |

### The vocabulary this lab installs (bilingual, once)

- **peeking / optional stopping (المراقبة المتكررة / التوقف الاختياري)** — looking repeatedly and stopping at the first significant result.
- **multiple testing (الاختبارات المتعددة)** — every extra metric, arm and segment is another chance for noise to win.
- **FWER / family-wise error rate (معدل الخطأ على مستوى العائلة)** — P(any false positive); controlled by Bonferroni.
- **FDR / false discovery rate (معدل الاكتشافات الخاطئة)** — the expected share of false positives among discoveries; controlled by Benjamini–Hochberg.
- **clustered standard errors (الأخطاء المعيارية العنقودية)** — required when the analysis unit is finer than the randomisation unit.
- **practical significance (الدلالة العملية)** — big enough to act on, which is a different question from p < 0.05.

> All code, variables and comments stay in **English**.


---
## Step 0 — Open the frozen plan *before* you open the results

This is the discipline the whole module rests on. The plan was hashed in Lab 2 and has
not been touched since. Read it first, so that when a segment result gets exciting in
Step 5 you already know what you promised to do about it.


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `causal_utils` importable from anywhere under course_assets/
ROOT = Path.cwd()
while not (ROOT / "causal_utils").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from causal_utils import use_sdaia_style
use_sdaia_style()

RANDOM_SEED = 213          # fixed for every lab: reproducibility is graded
rng = np.random.default_rng(RANDOM_SEED)
print("Environment ready ·", ROOT)

In [ ]:
def needs(*values, msg="complete the TODO in the cell above, then re-run this cell"):
    """Guard for the start notebook: True (with a friendly note) if a TODO is unfinished."""
    if any(v is None for v in values):
        print(f"[not implemented yet] {msg}")
        return True
    return False

In [ ]:
import json
from scipy import stats
from causal_utils import (srm_check, two_proportion_test, regression_effect,
                          correct_pvalues, peeking_simulation, cuped_adjust,
                          practical_significance_verdict, decision_plot, decision_memo,
                          AnalysisPlan, freeze_plan,
                          NAVY, BLUE, ORANGE, TEAL, SLATE, MUTED, PANEL)

with open(DATA / "GROUND_TRUTH.json") as f:
    GROUND_TRUTH = json.load(f)

plan_path = ROOT / "notebooks" / "lab2_frozen_plan.json"
if plan_path.exists():
    PLAN = json.loads(plan_path.read_text())
else:
    # Lab 2 was not run in this environment — rebuild the same plan so Lab 4 stands alone.
    PLAN = json.loads(freeze_plan(AnalysisPlan(
        experiment_name="injaz-guided-uploader-2026q2",
        decision="Ship the guided document uploader to all Injaz users?",
        randomisation_unit="user (national-ID hash)",
        oec="service-completion rate among triggered users",
        guardrails=["time-to-complete (session_duration_sec)", "support-contact rate",
                    "upload-error rate (error_flag)"],
        trigger_definition="user reaches the document-upload step at least once",
        unit_of_analysis="user", mde=0.02, alpha=0.05, power=0.80,
        correction_family=("no correction for the single OEC; Bonferroni across the "
                           "guardrail family; Benjamini-Hochberg for any segment sweep"),
        practical_threshold=0.01)).to_json())

print("FROZEN ANALYSIS PLAN (read before any outcome)")
print("=" * 66)
for k in ["experiment_name", "randomisation_unit", "unit_of_analysis", "oec",
          "trigger_definition", "mde", "alpha", "power", "practical_threshold",
          "correction_family", "stopping_rule", "plan_hash"]:
    print(f"  {k:<22}: {PLAN[k]}")
print("  guardrails            :")
for g in PLAN["guardrails"]:
    print(f"      - {g}")

THRESHOLD = PLAN["practical_threshold"]
ALPHA = PLAN["alpha"]

results = pd.read_csv(DATA / "injaz_experiment_results.csv")
print(f"\nassigned users in the results file : {len(results):,}")
print(f"columns                            : {list(results.columns)}")

---
## Step 1 *(8 min)* — The gate, then your first estimate

**The gate first.** Before any effect is computed, re-run the SRM check on the delivered
results. An SRM-failing experiment is not analysed — not "analysed with caveats", not
"analysed after weighting". Not analysed.

**Then the estimate.** You have 80,000 assigned users and a `completed` column. Compute
the lift. Do the obvious thing. We will come back to it.


In [ ]:
# --- the gate -------------------------------------------------------------
srm = srm_check(results["variant"], {"control": 0.5, "treatment": 0.5})
print(srm)
assert srm.passed, "SRM FAILED — stop here and fix the pipeline. Do not analyse."
print("\nGate passed. Proceeding to the analysis.\n")


def my_two_proportion(df, outcome, group, treated_label="treatment", alpha=0.05):
    """TODO: difference in proportions with a Wald 95% interval.

    Note the deliberate asymmetry that reviewers check for:
      - the p-value uses the POOLED variance (under H0 the two rates are equal)
      - the confidence interval uses the UNPOOLED variance (we are not assuming H0)

        p1, p0 = treated mean, control mean;  diff = p1 - p0
        p_pool = (successes_t + successes_c) / (n1 + n0)
        se_pool  = sqrt(p_pool*(1-p_pool)*(1/n1 + 1/n0))     -> z, then p-value
        se_unpool= sqrt(p1*(1-p1)/n1 + p0*(1-p0)/n0)         -> CI half-width

    Return a dict: estimate, se, ci_low, ci_high, p_value, n_treatment, n_control.
    Self-check: it must match causal_utils.two_proportion_test to 6 decimals.
    """
    return None


_probe = my_two_proportion(results, "completed", "variant")

if not needs(_probe, msg="implement my_two_proportion"):
    ref = two_proportion_test(results, "completed", "variant", label="ALL assigned users")
    assert abs(_probe["estimate"] - ref.estimate) < 1e-9, "point estimate disagrees"
    assert abs(_probe["ci_low"] - ref.ci_low) < 1e-6, "CI disagrees — check pooled vs unpooled"
    est_all = ref
    print(est_all)
    print(f"\nCompletion rate, control   : "
          f"{results.loc[results.variant == 'control', 'completed'].mean():.4f}")
    print(f"Completion rate, treatment : "
          f"{results.loc[results.variant == 'treatment', 'completed'].mean():.4f}")
    print("\nSignificant, positive, on 80,000 users. Write this number down — we will")
    print("find out in Step 2 what is wrong with it.")
else:
    est_all = None

---
## Step 2 *(8 min)* — Trap 1: you analysed the wrong population

Go back to the frozen plan and read the trigger definition:

> *"user reaches the document-upload step at least once"*

A user who logged in, paid a fee and never touched a document **could not possibly have
been affected by a change to the document uploader**. They are in your denominator anyway,
contributing pure noise and a guaranteed zero effect.

This is not a rounding issue. The measured effect is diluted by roughly the trigger rate:

$$\hat\tau_{\text{all assigned}} \approx \text{trigger rate} \times \hat\tau_{\text{triggered}}$$

Compute the trigger rate, re-estimate on triggered users only, and compare. Then run the
same thing as an OLS regression with HC3 robust standard errors — the point estimate must
be **identical**, because a regression on a single binary indicator *is* a difference in
means. What the regression buys you is Step 3.


In [ ]:
# TODO: compute the trigger rate and restrict to triggered users, then
# re-estimate. Use causal_utils.two_proportion_test and causal_utils.regression_effect.
#
#   trigger_rate = results["triggered"].mean()
#   triggered    = results.loc[results["triggered"] == 1].copy()
#   est_trig     = two_proportion_test(triggered, "completed", "variant", label=...)
#   est_ols      = regression_effect(triggered, "completed", "variant", label=...)

trigger_rate = None
triggered = None
est_trig = None
est_ols = None

if not needs(trigger_rate, triggered, est_trig, est_ols, est_all,
             msg="restrict to triggered users and re-estimate"):
    assert abs(est_trig.estimate - est_ols.estimate) < 1e-9, \
        "two-proportion and OLS point estimates MUST be identical — check your reference level"
    assert est_trig.estimate > est_all.estimate, \
        "the triggered estimate should be LARGER: the all-assigned one is diluted"
    print(f"trigger rate                     : {trigger_rate:.4f}")
    print(f"triggered users (the denominator): {len(triggered):,}\n")
    print(est_all); print(); print(est_trig); print(); print(est_ols)
    print("\nThe dilution arithmetic:")
    print(f"  triggered x trigger rate = {est_trig.estimate * trigger_rate:.4f}")
    print(f"  all-assigned estimate    = {est_all.estimate:.4f}")
    print("\nWrite one sentence: what question does the all-assigned number actually answer?")

In [ ]:
if needs(est_trig, msg="finish Step 2 to draw this"):
    pass
else:
    fig, ax = plt.subplots(figsize=(9.5, 4.2))
    labels = ["ALL 80,000 assigned users\n(the trap)", "49,653 TRIGGERED users\n(the frozen OEC)"]
    ests = [est_all, est_trig]
    colours = [MUTED, TEAL]
    for i, (e, colour) in enumerate(zip(ests, colours)):
        y = 1 - i
        ax.plot([100 * e.ci_low, 100 * e.ci_high], [y, y], color=colour, lw=5,
                solid_capstyle="round")
        ax.plot([100 * e.estimate], [y], "o", color=colour, ms=12,
                markeredgecolor="white", markeredgewidth=1.8)
        ax.text(100 * e.ci_high + 0.09, y, f"{100*e.estimate:+.2f} pp "
                f"[{100*e.ci_low:+.2f}, {100*e.ci_high:+.2f}]",
                va="center", fontsize=10, color=NAVY, fontweight="bold")

    ax.axvline(0, color=MUTED, lw=1.1)
    ax.axvline(100 * THRESHOLD, color=ORANGE, ls="--", lw=1.6,
               label=f"practical threshold ({100*THRESHOLD:.1f} pp)")
    ax.set_yticks([0, 1]); ax.set_yticklabels(labels[::-1])
    ax.set_xlabel("Effect on service-completion rate (percentage points)")
    ax.set_title("Trap 1: analysing everyone dilutes the effect toward zero")
    ax.set_xlim(-0.2, 4.6)
    ax.legend(loc="lower right")
    ax.grid(axis="y", visible=False)
    ax.annotate(
        f"Takeaway: both intervals exclude zero, so both look like a 'win'. Only one of them\n"
        f"is the effect of the feature. Diluting by the {trigger_rate:.0%} trigger rate cost "
        f"{100*(est_trig.estimate - est_all.estimate):.2f} pp\nof measured effect — enough, on a "
        "smaller experiment, to hide a real result entirely.",
        xy=(0.03, 0.06), xycoords="axes fraction", fontsize=10, color=SLATE,
        bbox=dict(boxstyle="round,pad=0.55", fc=PANEL, ec=MUTED, lw=0.8))
    plt.tight_layout()
    plt.show()

### ✅ Checkpoint A — what you should be seeing by now

≈16 minutes in. SRM passing, an all-assigned estimate around +1.34 pp, and a triggered
estimate around +2.16 pp with identical two-proportion and OLS point estimates.


In [ ]:
def checkpoint_a():
    if est_trig is None:
        print("✗ Steps 1-2 incomplete.")
        return
    print(f"✓ SRM gate               : {'PASS' if srm.passed else 'FAIL'} "
          f"(chi2 {srm.chi2:.2f}, p {srm.p_value:.3f})")
    print(f"✓ all assigned (diluted) : {est_all.as_pp()}")
    print(f"✓ triggered (the OEC)    : {est_trig.as_pp()}")
    print(f"✓ OLS + HC3 agrees       : {abs(est_trig.estimate - est_ols.estimate) < 1e-9}")
    print(f"✓ trigger rate           : {trigger_rate:.4f}")
    print()
    print("Next: the same estimate with a tighter interval, then the guardrails.")

checkpoint_a()

---
## Step 3 *(8 min)* — Same estimate, tighter interval

Lab 3 measured a 14% variance reduction from `prior_completion_rate`. Now spend it.

Adding a **pre-treatment** covariate to the regression cannot bias a randomised
comparison — treatment is independent of it by construction — but it absorbs residual
variance and narrows the interval. That is the entire CUPED bargain, and the one
condition that makes it safe is that the covariate was measured *before* assignment.

> ⚠️ Never put a **post-treatment** variable in the covariate list. `support_contact`,
> `session_duration_sec` and `error_flag` are all downstream of the treatment. Adjusting
> for them is Module 6's mediator/collider trap and it will quietly destroy the estimate.
> The cell after the good one demonstrates exactly that.


In [ ]:
# TODO: add the pre-treatment covariate to the regression.
#
#   est_cuped = regression_effect(triggered, "completed", "variant",
#                                 covariates=["prior_completion_rate"], label=...)
est_cuped = None

if not needs(est_cuped, est_trig, msg="add the pre-period covariate"):
    width_plain = est_trig.ci_high - est_trig.ci_low
    width_cuped = est_cuped.ci_high - est_cuped.ci_low
    assert width_cuped < width_plain, \
        "the interval must get NARROWER — if it did not, your covariate is not predictive"
    assert abs(est_cuped.estimate - est_trig.estimate) < 0.005, \
        "the point estimate should barely move: adjustment cuts variance, not bias"
    print(est_trig); print(); print(est_cuped)
    print(f"\ninterval narrowed by {1 - width_cuped / width_plain:.1%}   "
          f"(variance reduction {1 - (est_cuped.se / est_trig.se)**2:.1%})")

    # TODO (discussion): now add "drop_off" to the covariate list — a variable
    # measured AFTER assignment — and look at what happens to the estimate.
    # Then explain in one sentence why that is NOT "controlling for more".
    bad = None
    if not needs(bad, msg="run the post-treatment-covariate demonstration"):
        print(bad)

---
## Step 4 *(8 min)* — The guardrail family, under Bonferroni

A feature that lifts completion while doubling support load has not succeeded. The frozen
plan names three guardrails, and the frozen correction family for them is **Bonferroni**:
a guardrail breach is expensive, so you want to control the probability of **any** false
alarm, not the expected proportion of them.

Note the asymmetry in how you read guardrails. For the OEC you ask "is there an effect?".
For a guardrail you ask "can I rule out a harmful effect large enough to matter?" — a
question about the interval, not the p-value.


In [ ]:
# TODO: run the guardrail family.
#
#   1. For the three BINARY guardrails (support_contact, error_flag, drop_off) use
#      two_proportion_test on `triggered`.
#   2. For time-to-complete (session_duration_sec) use a Welch t-test — it is a mean,
#      not a proportion: stats.ttest_ind(t, c, equal_var=False).
#   3. Collect the four raw p-values and apply correct_pvalues(..., method="bonferroni").
#   4. Print effect, raw p and adjusted p for each, and state whether any guardrail moved.
#
# Self-check: with m = 4, a raw p of ~0.04 must NOT survive.

gdf = None

if not needs(gdf, triggered, msg="build the guardrail table"):
    assert len(gdf) == 4, "the frozen plan names three guardrails plus time-to-complete"
    assert "p_adjusted" in gdf.columns, "apply the Bonferroni correction"
    print(gdf.to_string(index=False))
    print("\nWrite one sentence per guardrail: did it move, and does the move MATTER?")

---
## Step 5 *(8 min)* — The segment sweep, and the segment that vanishes

Now the exploratory sweep: 13 regions × devices, about 16 tests. Under the null, 16 tests
at α = 0.05 give a **56%** chance of at least one false positive. Every extra segment is
another lottery ticket.

The frozen plan says the sweep gets **Benjamini–Hochberg**: this is an exploratory
hypothesis list, not a guardrail family, so controlling the *expected proportion* of false
discoveries is the right trade — more power than Bonferroni, at the cost of tolerating
some false discoveries.

Run it. Get excited about a segment. Then correct it.


In [ ]:
# TODO: run the segment sweep.
#
#   For each level of "region" and "device" with at least 200 triggered users and both
#   arms present, run two_proportion_test on `completed`, and collect
#   segment, n, estimate, ci_low, ci_high, p_raw.
#   Then apply correct_pvalues(..., method="bh") and merge the adjusted p-values in.

sweep = None

if not needs(sweep, msg="run the segment sweep and apply BH"):
    assert "p_adjusted" in sweep.columns, "apply the Benjamini-Hochberg correction"
    raw_sig = int((sweep["p_raw"] < ALPHA).sum())
    survive = int(sweep["reject"].sum())
    assert survive < raw_sig, "BH must kill some of the raw-significant segments"
    print(f"tests in the family : {len(sweep)}")
    print(f"P(at least one false positive if NOTHING is real) = "
          f"{1 - 0.95**len(sweep):.0%}\n")
    print(sweep.sort_values("p_raw").to_string(index=False))
    print(f"\nraw-significant : {raw_sig}    surviving BH : {survive}")
    print("\nWrite two sentences: why is a surviving segment still only a hypothesis?")

In [ ]:
if needs(sweep, est_trig, msg="finish Step 5 to draw this"):
    pass
else:
    fig, ax = plt.subplots(figsize=(10, 6.5))
    s = sweep.sort_values("estimate").reset_index(drop=True)
    for i, r in s.iterrows():
        colour = TEAL if r["reject"] else (BLUE if r["p_raw"] < ALPHA else MUTED)
        ax.plot([100 * r["ci_low"], 100 * r["ci_high"]], [i, i], color=colour, lw=2.6,
                solid_capstyle="round")
        ax.plot([100 * r["estimate"]], [i], "o", color=colour, ms=7,
                markeredgecolor="white", markeredgewidth=1.2)

    ax.axvline(0, color=MUTED, lw=1.1)
    ax.axvline(100 * est_trig.estimate, color=NAVY, ls="--", lw=1.7,
               label=f"platform-wide effect ({100*est_trig.estimate:+.2f} pp)")
    ax.set_yticks(range(len(s)))
    ax.set_yticklabels(s["segment"], fontsize=9)
    ax.set_xlabel("Effect on completion rate within the segment (percentage points)")
    ax.set_title("Segment sweep: 16 tests, and what survives Benjamini–Hochberg")
    ax.legend(loc="lower right")
    ax.grid(axis="y", visible=False)
    ax.annotate(
        f"Takeaway: {int((sweep['p_raw'] < ALPHA).sum())} segments are raw-significant; "
        f"{int(sweep['reject'].sum())} survive BH (teal).\n"
        "Every interval overlaps the platform-wide dashed line — there is no evidence\n"
        "of real heterogeneity here, only of small samples and 16 lottery tickets.",
        xy=(0.03, 0.05), xycoords="axes fraction", fontsize=10, color=SLATE,
        bbox=dict(boxstyle="round,pad=0.55", fc=PANEL, ec=MUTED, lw=0.8))
    plt.tight_layout()
    plt.show()

---
## Step 6 *(8 min)* — Trap 2: the unit-of-analysis mismatch

Open `injaz_sessions.csv.gz`: 339,017 sessions from the same 80,000 users, about 4.2
sessions each. A colleague analyses completion **per session**, because "more rows means
more power".

More rows, yes. More *information*, no. Sessions within a user are correlated — the same
person, the same device, the same habits — so treating 339,017 sessions as independent
observations of a **user-randomised** experiment produces a standard error that is far too
small and a confidence interval that lies about precision.

Three analyses, one truth:

1. **Session-level, no clustering** — the seductive, wrong one.
2. **Session-level, clustered by `user_id`** — honest.
3. **Aggregated to the user, then analysed** — also honest, and simpler to explain.

The point estimate barely moves. The interval is the whole story.


In [ ]:
sessions = pd.read_csv(DATA / "injaz_sessions.csv.gz")
print(f"sessions            : {len(sessions):,}")
print(f"distinct users      : {sessions['user_id'].nunique():,}")
print(f"sessions per user   : {len(sessions) / sessions['user_id'].nunique():.2f}\n")

# TODO: run the SAME comparison three ways and put the standard errors side by side.
#
#   naive_sess = two_proportion_test(sessions, "session_completed", "variant", label=...)
#   clustered  = regression_effect(sessions, "session_completed", "variant",
#                                  cluster="user_id", label=...)
#   user_level = sessions.groupby(["user_id", "variant"], as_index=False)["session_completed"].mean()
#   aggregated = regression_effect(user_level, "session_completed", "variant", label=...)

naive_sess = None
clustered = None
aggregated = None

if not needs(naive_sess, clustered, aggregated, msg="run the three session-level analyses"):
    assert clustered.se > naive_sess.se, \
        "clustering must WIDEN the standard error — check the cluster argument"
    assert abs(clustered.estimate - naive_sess.estimate) < 1e-9, \
        "the point estimate must not move; only the uncertainty changes"
    print(f"{'analysis':<42}{'estimate':>11}{'SE':>10}{'CI width':>12}")
    print("-" * 78)
    for e in (naive_sess, clustered, aggregated):
        print(f"{e.label:<42}{100*e.estimate:>10.2f}pp{e.se:>10.5f}"
              f"{100*(e.ci_high - e.ci_low):>11.2f}pp")
    print(f"\nIgnoring clustering understates the SE by a factor of "
          f"{clustered.se / naive_sess.se:.2f}x.")
    print("Write one sentence: what did NOT change, and why does that make the bug dangerous?")

In [ ]:
if needs(clustered, msg="finish Step 6 to draw this"):
    pass
else:
    fig, ax = plt.subplots(figsize=(9.5, 4.4))
    items = [(naive_sess, "#C0392B", "session-level, NO clustering\n(falsely narrow — wrong)"),
             (clustered, TEAL, "session-level, clustered by user\n(honest width)"),
             (aggregated, BLUE, "aggregated to the user\n(honest, and simpler)")]
    for i, (e, colour, label) in enumerate(items):
        y = len(items) - 1 - i
        ax.plot([100 * e.ci_low, 100 * e.ci_high], [y, y], color=colour, lw=5,
                solid_capstyle="round")
        ax.plot([100 * e.estimate], [y], "o", color=colour, ms=11,
                markeredgecolor="white", markeredgewidth=1.6)
        ax.text(100 * e.ci_high + 0.04, y,
                f"[{100*e.ci_low:+.2f}, {100*e.ci_high:+.2f}]  width "
                f"{100*(e.ci_high - e.ci_low):.2f} pp",
                va="center", fontsize=9.5, color=NAVY)

    ax.axvline(0, color=MUTED, lw=1.1)
    ax.set_yticks(range(len(items)))
    ax.set_yticklabels([lbl for _, _, lbl in items][::-1], fontsize=9.5)
    ax.set_xlabel("Effect on session-completion rate (percentage points)")
    ax.set_title("Trap 2: 339,017 sessions are not 339,017 independent observations")
    ax.set_xlim(0.2, 2.4)
    ax.grid(axis="y", visible=False)
    ax.annotate(
        f"Takeaway: the point estimate is identical in all three. Ignoring clustering shrinks\n"
        f"the interval to {100*(naive_sess.ci_high - naive_sess.ci_low):.2f} pp when the honest "
        f"width is {100*(clustered.ci_high - clustered.ci_low):.2f} pp — a "
        f"{clustered.se / naive_sess.se:.1f}x overstatement of precision\n"
        "that no amount of extra data will correct, because the extra rows are the problem.",
        xy=(0.03, 0.05), xycoords="axes fraction", fontsize=10, color=SLATE,
        bbox=dict(boxstyle="round,pad=0.55", fc=PANEL, ec=MUTED, lw=0.8))
    plt.tight_layout()
    plt.show()

### ✅ Checkpoint B — what you should be seeing by now

≈40 minutes in. Guardrails corrected, a segment sweep where most raw-significant results
evaporate, and the clustered interval about 1.6× wider than the naive one.


In [ ]:
def checkpoint_b():
    if est_cuped is None or sweep is None or clustered is None or gdf is None:
        print("✗ Steps 3-6 incomplete.")
        return
    print(f"✓ CUPED-adjusted OEC     : {est_cuped.as_pp()}   "
          f"(SE {est_trig.se:.5f} -> {est_cuped.se:.5f})")
    print(f"✓ guardrails corrected   : {len(gdf)} metrics, Bonferroni, "
          f"{int(gdf['reject'].sum())} moved")
    print(f"✓ segment sweep          : {len(sweep)} tests, "
          f"{int((sweep['p_raw'] < ALPHA).sum())} raw-significant, "
          f"{int(sweep['reject'].sum())} survive BH")
    print(f"✓ clustering penalty     : SE x{clustered.se / naive_sess.se:.2f}")
    print()
    print("Next: the peeking simulation, then the decision.")

checkpoint_b()

---
## Step 7 *(10 min)* — Peeking: how 5% becomes 25%

A fixed-horizon test controls α **only if you look once**, at the pre-computed sample
size. Every additional peek is another chance for noise to cross the threshold, and the
cumulative false-positive rate compounds.

The simulation below runs experiments where the true effect is **exactly zero**. Every
"significant" result is therefore a false positive, by construction. The only thing that
changes across rows is how often you looked.

Three valid responses, and you must pick one *before* launch:

- **Fixed-horizon discipline** — compute n, run to it, analyse once.
- **Group-sequential methods** (O'Brien–Fleming, Pocock) — a small number of pre-planned
  interim looks with spending-function-adjusted thresholds.
- **Always-valid inference** — confidence sequences valid at every moment, so continuous
  monitoring is legitimate. The price is a wider interval early on.

The instinct to install: **if you want to watch continuously, you must use a method built
for continuous watching.**


In [ ]:
def my_peeking_simulation(n_looks_grid=(1, 5, 10, 20), n_per_arm=5_000,
                          p=0.55, n_experiments=2_000, alpha=0.05, seed=RANDOM_SEED):
    """TODO: simulate optional stopping under a TRUE NULL and count false positives.

    Both arms come from the SAME Bernoulli(p), so the true effect is ZERO and every
    significant result is a false positive.

    Efficient recipe (a naive double loop takes minutes; this takes seconds):
      a = g.binomial(1, p, size=(n_experiments, n_per_arm))     # arm A
      b = g.binomial(1, p, size=(n_experiments, n_per_arm))     # arm B
      ca, cb = np.cumsum(a, axis=1), np.cumsum(b, axis=1)
      for n_looks in n_looks_grid:
          idx = np.linspace(n_per_arm / n_looks, n_per_arm, n_looks).astype(int) - 1
          for i in idx:                                # each inspection point
              n = i + 1
              p1, p0  = ca[:, i] / n, cb[:, i] / n
              p_pool  = (ca[:, i] + cb[:, i]) / (2 * n)
              se      = sqrt(p_pool * (1 - p_pool) * 2 / n)
              mark experiments where |(p1 - p0) / se| > norm.ppf(1 - alpha/2)
          record the SHARE of experiments significant at ANY look

    Return a DataFrame with n_looks, false_positive_rate, inflation_vs_nominal.

    Self-check: ~5% at 1 look rising to ~25% at 20 looks.
    """
    return None


peek = my_peeking_simulation()

if not needs(peek, msg="implement my_peeking_simulation"):
    one = float(peek.loc[peek.n_looks == 1, "false_positive_rate"].iloc[0])
    twenty = float(peek.loc[peek.n_looks == 20, "false_positive_rate"].iloc[0])
    assert 0.03 < one < 0.07, f"one look must reproduce alpha (~5%), got {one:.1%}"
    assert twenty > 4 * one, \
        f"twenty looks must inflate the error rate several-fold, got {twenty:.1%}"
    print("2,000 experiments, TRUE effect = 0, alpha = 0.05\n")
    print(f"{'looks':>7}{'false-positive rate':>24}{'inflation vs 5%':>19}")
    print("-" * 52)
    for _, r in peek.iterrows():
        print(f"{int(r['n_looks']):>7}{r['false_positive_rate']:>23.1%}"
              f"{r['inflation_vs_nominal']:>18.1f}x")
    print("\ncausal_utils cross-check:")
    print(peeking_simulation(n_looks_grid=(1, 5, 10, 20), n_per_arm=5_000,
                             p=0.55, n_experiments=2_000, seed=RANDOM_SEED).to_string(index=False))
    print("\nWrite the ONE-SENTENCE rule this table justifies.")

In [ ]:
if needs(peek, msg="finish Step 7 to draw this"):
    pass
else:
    fine = my_peeking_simulation(n_looks_grid=(1, 2, 3, 5, 8, 10, 14, 20),
                                 n_experiments=2_000)

    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    ax.step(fine["n_looks"], 100 * fine["false_positive_rate"], where="mid",
            color=ORANGE, lw=2.6, label="actual false-positive rate")
    ax.plot(fine["n_looks"], 100 * fine["false_positive_rate"], "o", color=ORANGE, ms=7,
            markeredgecolor="white", markeredgewidth=1.4)
    ax.axhline(5, color=TEAL, ls="--", lw=1.8, label="the 5% you think you are getting")

    for _, r in fine.iterrows():
        if int(r["n_looks"]) in (1, 5, 10, 20):
            ax.annotate(f"{100*r['false_positive_rate']:.0f}%",
                        xy=(r["n_looks"], 100 * r["false_positive_rate"]),
                        xytext=(r["n_looks"] + 0.35, 100 * r["false_positive_rate"] + 1.1),
                        fontsize=10, color=NAVY, fontweight="bold")

    ax.set_xlabel("Number of times you looked at the dashboard before deciding")
    ax.set_ylabel("Probability of declaring a winner when NOTHING is real (%)")
    ax.set_title("Peeking: the false-positive rate you actually run")
    ax.legend(loc="lower right")
    ax.set_ylim(0, 30)
    ax.annotate(
        "Takeaway: every experiment here has a true effect of exactly zero. Checking the\n"
        "dashboard daily for three weeks turns a 5% error rate into roughly 25% — one in four\n"
        "'wins' is noise. This is a property of the STOPPING RULE, not of the data.",
        xy=(0.04, 0.62), xycoords="axes fraction", fontsize=10, color=SLATE,
        bbox=dict(boxstyle="round,pad=0.55", fc=PANEL, ec=MUTED, lw=0.8))
    plt.tight_layout()
    plt.show()

---
## Step 8 *(8 min)* — Statistical significance is not a decision

You now have an effect, an honest interval, clean guardrails and no surviving segment
story. One question remains, and it is not a statistical one:

**Is the effect big enough to be worth shipping?**

The frozen plan set a **practical-significance threshold** of 1.0 pp — the smallest lift
that justifies the build and maintenance cost. Map the *interval* onto the decision, not
the p-value onto a verdict. There are four cases, and only one of them is "ship".


In [ ]:
# TODO: turn the estimate into a decision.
#
#   1. print(practical_significance_verdict(est_cuped, THRESHOLD, metric_name=...))
#   2. decision_plot([est_all, est_trig, est_cuped, clustered], threshold=THRESHOLD, ...)
#   3. Write the decision paragraph below. It MUST contain:
#        - the lift and its 95% confidence interval, in percentage points
#        - the guardrail verdict, including any move you are choosing to ACCEPT
#        - an explicit statistical-vs-practical significance statement
#        - what you are NOT claiming

if not needs(est_cuped, msg="finish Step 3 first"):
    print(practical_significance_verdict(est_cuped, THRESHOLD,
                                         metric_name="Completion rate (triggered users)"))

DECISION_PARAGRAPH = """
TODO: write it here.
"""
print(DECISION_PARAGRAPH)

---
## Step 9 — Grade yourself against the planted ground truth

The uploader effect was planted at **+2.2 pp among triggered users**. Nothing in this
notebook was tuned to find it. Compare.


In [ ]:
if needs(est_trig, est_cuped, est_all, est_ols, clustered, naive_sess,
         peek, trigger_rate, msg="finish Steps 1-7 first"):
    pass
else:
    M4 = GROUND_TRUTH["module4_uploader_experiment"]

    print(f"{'quantity':<46}{'recovered':>12}{'planted':>12}   verdict")
    print("-" * 84)
    rows = [
        ("effect on TRIGGERED users", est_trig.estimate, M4["true_lift_on_exposed"], 0.003),
        ("effect, CUPED-adjusted", est_cuped.estimate, M4["true_lift_on_exposed"], 0.003),
        ("effect on ALL assigned (diluted)", est_all.estimate, M4["true_lift_on_exposed"], 0.003),
        ("trigger rate", trigger_rate, M4["trigger_rate"], 0.005),
        ("treatment allocation", float((results.variant == "treatment").mean()),
         M4["allocation_treatment"], 0.002),
    ]
    for name, got, want, tol in rows:
        print(f"{name:<46}{got:>12.4f}{want:>12.4f}   "
              f"{'MATCH' if abs(got - want) <= tol else 'MISSED — see below'}")

    print()
    print(f"The triggered estimate lands {abs(est_trig.estimate - M4['true_lift_on_exposed'])*100:.3f} pp "
          "from the planted truth.")
    print(f"The all-assigned estimate misses it by "
          f"{abs(est_all.estimate - M4['true_lift_on_exposed'])*100:.2f} pp — and its 95% interval "
          f"[{100*est_all.ci_low:+.2f}, {100*est_all.ci_high:+.2f}] does not even contain +2.20 pp.")
    print("That is what a population error costs. It is not noise, and it does not shrink.")
    print()
    print("Benchmarks for Module 4:")
    print(f"  two-prop == OLS point estimate to 4 dp : "
          f"{abs(est_trig.estimate - est_ols.estimate) < 1e-4}")
    print(f"  clustered SE wider than naive          : "
          f"{clustered.se > naive_sess.se}   ({clustered.se / naive_sess.se:.2f}x)")
    print(f"  peeking FPR at 20 looks ~25%           : "
          f"{float(peek.loc[peek.n_looks == 20, 'false_positive_rate'].iloc[0]):.1%}")
    print(f"  correction families applied            : Bonferroni (guardrails), BH (sweep)")

### Benchmark table for Module 4 (fill this in from your own run)

| Analysis | Estimate | 95% CI | Verdict |
|---|---|---|---|
| All assigned users (ITT) | | | |
| Triggered, two-proportion | | | |
| Triggered, CUPED-adjusted | | | |
| Session-level, no clustering | | | falsely narrow — wrong |
| Session-level, clustered | | | honest width |


---
## What you now own

In [ ]:
print("""
TOOLKIT CONTRIBUTION — Lab 4
============================================================
Reusable artefacts you built and can lift into any project:

  The analysis GATE
      srm_check first, always. An SRM-failing experiment is not analysed.

  my_two_proportion(df, outcome, group)
      Difference in proportions with the pooled/unpooled asymmetry done
      right: pooled variance for the p-value, unpooled for the interval.

  The TRIGGERED-population discipline
      Read the frozen trigger definition before choosing a denominator.
      Analysing everyone dilutes a real effect by the trigger rate.

  Covariate-adjusted estimation (regression_effect + a PRE-treatment covariate)
      The same estimate with a 14% tighter variance - and the demonstration
      of what post-treatment covariates do to it.

  The correction-family decision
      One pre-registered OEC -> no correction.
      A guardrail family      -> Bonferroni (FWER).
      An exploratory sweep    -> Benjamini-Hochberg (FDR), and a survivor is
                                 still only a hypothesis.

  Cluster-robust analysis (regression_effect(..., cluster="user_id"))
      Mandatory whenever the analysis row is finer than the randomisation
      unit. The point estimate never moves; the honesty does.

  my_peeking_simulation(...)
      The 5% -> 25% exhibit. Run it live the next time someone screenshots a
      p = 0.03 on day three.

  The decision memo (decision_plot + decision_memo)
      Effect, interval, threshold, guardrails, and what you are NOT claiming.
      No p-values reach this page.

Carried forward:
  - Labs 5-6 lose randomisation entirely and must argue for identification.
  - The capstone reruns this exact pipeline on the autofill experiment, where
    the interval STRADDLES the 1.0 pp threshold and the right answer is
    'hold / extend', not 'ship'.
============================================================
""")

---
## Mini-exercises

### 1 · Quiz

1. Why does peeking inflate the false-positive rate?
2. When are clustered standard errors required?
3. Bonferroni vs Benjamini–Hochberg — which controls what, and when do you use each?
4. Why can a ratio metric not be t-tested as a mean?
5. Statistical vs practical significance — state the difference in one sentence each.

### 2 · Correction exercise

Eight secondary p-values: `[0.001, 0.011, 0.020, 0.033, 0.041, 0.049, 0.20, 0.55]`.
Apply Bonferroni and BH, by hand first and then in code. Which discoveries does each keep,
and why is the difference exactly what you would want for an exploratory family?

### 3 · The delta method — a ratio metric done properly

`completions per session` has a **random denominator**. A per-value t-test understates its
variance. Compute the effect on this ratio metric with the delta method and compare the
standard error against the naive one.

### 4 · Discussion

- A stakeholder screenshots a p = 0.03 on day 3 and wants to ship. What do you say, and
  what *would* let you legitimately act early?
- When is FDR the right choice over FWER for the Injaz metric families?


In [ ]:
# --- Mini-exercise 2: Bonferroni vs BH -----------------------------------
secondary = [0.001, 0.011, 0.020, 0.033, 0.041, 0.049, 0.20, 0.55]
bonf = correct_pvalues(secondary, method="bonferroni", alpha=0.05)
bh8 = correct_pvalues(secondary, method="bh", alpha=0.05)
merged = bonf[["metric", "p_raw", "p_adjusted", "reject"]].merge(
    bh8[["metric", "p_adjusted", "reject"]], on="metric",
    suffixes=("_bonferroni", "_bh"))
print("Eight secondary p-values, m = 8\n")
print(merged.to_string(index=False))
print(f"\nBonferroni keeps {int(merged['reject_bonferroni'].sum())} discovery; "
      f"BH keeps {int(merged['reject_bh'].sum())}.")
print("Bonferroni controls P(ANY false positive) and pays for it in power. BH controls")
print("the expected SHARE of false positives among discoveries — the right trade when the")
print("output is a hypothesis list rather than a set of go/no-go gates.")

# --- Mini-exercise 3: the delta method -----------------------------------
print("\n\n--- Ratio metric: completions per session ---\n")
per_user = sessions.groupby(["user_id", "variant"], as_index=False).agg(
    completions=("session_completed", "sum"), n_sessions=("session_id", "count"))


def ratio_delta(num, den):
    """Ratio estimate and its delta-method standard error."""
    num, den = np.asarray(num, float), np.asarray(den, float)
    n = len(num)
    mx, my = num.mean(), den.mean()
    vx, vy = num.var(ddof=1), den.var(ddof=1)
    cov = np.cov(num, den, ddof=1)[0, 1]
    ratio = mx / my
    var = (vx - 2 * (mx / my) * cov + (mx / my) ** 2 * vy) / (my ** 2 * n)
    return ratio, float(np.sqrt(var))


out = {}
for arm in ("treatment", "control"):
    sub = per_user.loc[per_user.variant == arm]
    out[arm] = ratio_delta(sub["completions"], sub["n_sessions"])

diff = out["treatment"][0] - out["control"][0]
se_delta = np.sqrt(out["treatment"][1] ** 2 + out["control"][1] ** 2)

# the naive alternative: treat each user's own ratio as an observation
per_user["user_ratio"] = per_user["completions"] / per_user["n_sessions"]
t_r = per_user.loc[per_user.variant == "treatment", "user_ratio"]
c_r = per_user.loc[per_user.variant == "control", "user_ratio"]
se_naive = np.sqrt(t_r.var(ddof=1) / len(t_r) + c_r.var(ddof=1) / len(c_r))

se_session = getattr(naive_sess, "se", None)   # from Step 6, if you ran it

print(f"ratio, treatment            : {out['treatment'][0]:.4f}")
print(f"ratio, control              : {out['control'][0]:.4f}")
print(f"difference                  : {diff:+.4f} completions per session\n")
print(f"{'variance estimator':<38}{'SE':>10}{'95% CI':>26}")
print("-" * 76)
print(f"{'delta method (user-clustered)':<38}{se_delta:>10.5f}"
      f"{f'[{diff - 1.96*se_delta:+.4f}, {diff + 1.96*se_delta:+.4f}]':>26}")
print(f"{'mean of per-user ratios':<38}{se_naive:>10.5f}"
      f"{f'[{diff - 1.96*se_naive:+.4f}, {diff + 1.96*se_naive:+.4f}]':>26}")
if se_session is not None:
    print(f"{'naive SESSION-level t-test (wrong)':<38}{se_session:>10.5f}"
          f"{f'[{diff - 1.96*se_session:+.4f}, {diff + 1.96*se_session:+.4f}]':>26}")
    print()
    print(f"The naive session-level interval is {se_delta / se_session:.1f}x too narrow: "
          "it treats")
    print("339,017 sessions as independent when there were only 80,000 coin flips.")
print()
print("The first two are both defensible and differ by ESTIMAND, not by correctness: the")
print("delta method targets E[completions] / E[sessions] (the platform ratio, denominator")
print("random), the mean-of-user-ratios weights a one-session user the same as a")
print("ten-session user. Decide which quantity the decision needs BEFORE you pick the")
print("estimator — most teams never notice they have silently chosen one.")